# LeetCode 680 — Valid Palindrome II bằng Python

**Link đề:** https://leetcode.com/problems/valid-palindrome-ii/
**Độ khó:** Easy · **Chủ đề:** Two Pointers, String, Greedy

```python
class Solution:
    def validPalindrome(self, s: str) -> bool:
        ...
```

## Mục tiêu học
1. Ôn lại palindrome và kiểm tra palindrome bằng `s[::-1]` và bằng **hai con trỏ**.
2. Viết brute force "thử xóa từng ký tự" và thấy vì sao `O(n²)` quá chậm với `n = 10^5`.
3. Hiểu insight **tham lam** (greedy): chỉ cần xử lý ở **cặp lệch đầu tiên** — thử bỏ trái **hoặc** bỏ phải.
4. Viết hàm phụ `is_pal_range(s, i, j)` kiểm tra đoạn con **không tạo chuỗi mới**.
5. Tổng quát hóa sang "xóa tối đa `k` ký tự" bằng đệ quy có ngân sách.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho chuỗi `s`. Trả `True` nếu `s` có thể trở thành **palindrome** sau khi xóa **tối đa một** ký tự.

**Ví dụ 1:** `s = "aba"` → `True` (đã là palindrome, không cần xóa).
**Ví dụ 2:** `s = "abca"` → `True` (xóa `'c'` → `"aba"`; hoặc xóa `'b'` → `"aca"`).
**Ví dụ 3:** `s = "abc"` → `False`.

**Constraints:** `1 <= len(s) <= 10^5`, `s` chỉ gồm chữ thường tiếng Anh.

"Tối đa một" = được xóa **0 hoặc 1** ký tự.

## 2. Nền tảng Python cần biết

### 2.1. Palindrome và `s[::-1]`
Palindrome đọc xuôi = đọc ngược. `s[::-1]` tạo chuỗi đảo (slicing với bước `-1`), tốn `O(n)` thời gian và bộ nhớ.

In [1]:
for s in ["aba", "abca", "racecar", "ab"]:
    print(f"{s!r:<10} đảo = {s[::-1]!r:<10} palindrome? {s == s[::-1]}")

'aba'      đảo = 'aba'      palindrome? True
'abca'     đảo = 'acba'     palindrome? False
'racecar'  đảo = 'racecar'  palindrome? True
'ab'       đảo = 'ba'       palindrome? False


### 2.2. Xóa một ký tự bằng slicing `s[:i] + s[i+1:]`
- `s[:i]`: từ đầu đến **trước** `i`.
- `s[i+1:]`: từ sau `i` đến hết.
Ghép lại = `s` bỏ ký tự ở vị trí `i`. Chuỗi Python **bất biến** nên luôn tạo chuỗi mới (`O(n)`).

In [2]:
s = "abca"
for i in range(len(s)):
    t = s[:i] + s[i + 1:]
    print(f"xóa s[{i}]={s[i]!r} -> {t!r}, palindrome? {t == t[::-1]}")

xóa s[0]='a' -> 'bca', palindrome? False
xóa s[1]='b' -> 'aca', palindrome? True
xóa s[2]='c' -> 'aba', palindrome? True
xóa s[3]='a' -> 'abc', palindrome? False


### 2.3. Hai con trỏ kiểm tra palindrome
`i` từ trái, `j` từ phải; so `s[i]` với `s[j]`; khớp thì `i += 1, j -= 1`; lệch thì không phải palindrome. Dừng khi `i >= j`.
Ưu điểm: không tạo chuỗi mới (`O(1)` bộ nhớ) và **dừng sớm** khi lệch.

In [3]:
def is_pal_two_pointers(s: str) -> bool:
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1
        j -= 1
    return True

print([is_pal_two_pointers(x) for x in ["aba", "abba", "abca", "a"]])

[True, True, False, True]


### 2.4. Kiểm tra **một đoạn** `s[i..j]` mà không cắt chuỗi
Viết hàm nhận thêm hai chỉ số biên — đây sẽ là "viên gạch" của lời giải chính.

In [4]:
def is_pal_range(s: str, i: int, j: int) -> bool:
    # s[i..j] (bao gồm cả hai đầu) có phải palindrome không?
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1
        j -= 1
    return True

s = "xabay"
print(is_pal_range(s, 1, 3), "-> đoạn", s[1:4])
print(is_pal_range(s, 0, 4), "-> cả chuỗi", s)

True -> đoạn aba
False -> cả chuỗi xabay


## 3. Cách trực giác — Brute Force

1. Nếu `s` đã là palindrome → `True`.
2. Thử xóa **từng** vị trí `i`, kiểm tra chuỗi còn lại.

Có `n` cách xóa, mỗi lần tạo chuỗi + kiểm tra `O(n)` → `O(n²)`. Với `n = 10^5` khoảng `10^10` thao tác → TLE.

In [5]:
def valid_palindrome_bruteforce(s: str) -> bool:
    if s == s[::-1]:
        return True
    for i in range(len(s)):
        t = s[:i] + s[i + 1:]
        if t == t[::-1]:
            return True
    return False

for s in ["aba", "abca", "abc", "deeee", "eeccccbebaeeabebccceea"]:
    print(f"{s!r:<26} -> {valid_palindrome_bruteforce(s)}")

'aba'                      -> True
'abca'                     -> True
'abc'                      -> False
'deeee'                    -> True
'eeccccbebaeeabebccceea'   -> False


## 4. Nút thắt và insight — chỉ cặp lệch **đầu tiên** quan trọng

Chạy hai con trỏ từ hai đầu. Các cặp **khớp** ở ngoài không bao giờ cần xóa (xóa chúng chỉ làm hỏng đối xứng).

Gọi `(i, j)` là **cặp lệch đầu tiên**: `s[i] != s[j]`. Mọi cách xóa 1 ký tự thành công **phải xóa `s[i]` hoặc `s[j]`** (nếu xóa ký tự khác, cặp `(i, j)` vẫn đối diện nhau và vẫn lệch).

Vậy chỉ có **2 khả năng**:
- bỏ `s[i]` → kiểm tra `s[i+1..j]` có palindrome không;
- bỏ `s[j]` → kiểm tra `s[i..j-1]`.

Mỗi kiểm tra `O(n)` → tổng `O(n)`.

```text
s = "a b c a"      i=0,j=3: a==a khớp
       ^   ^       i=1,j=2: b!=c LỆCH
thử bỏ 'b': "c"    -> palindrome ✔
```

In [6]:
s = "abca"
i, j = 0, len(s) - 1
while i < j and s[i] == s[j]:
    print(f"khớp: s[{i}]={s[i]} == s[{j}]={s[j]}")
    i += 1; j -= 1
print(f"cặp lệch đầu tiên: i={i} ({s[i]}), j={j} ({s[j]})")
print("bỏ trái -> đoạn", repr(s[i + 1:j + 1]), is_pal_range(s, i + 1, j))
print("bỏ phải -> đoạn", repr(s[i:j]), is_pal_range(s, i, j - 1))

khớp: s[0]=a == s[3]=a
cặp lệch đầu tiên: i=1 (b), j=2 (c)
bỏ trái -> đoạn 'c' True
bỏ phải -> đoạn 'b' True


## 5. Lời giải chính — Hai con trỏ + thử 2 nhánh (giảng từng đoạn)

**Đoạn 1:** `i, j = 0, len(s) - 1`.

**Đoạn 2:** trong khi `i < j`:
- `s[i] == s[j]` → thu hẹp `i += 1, j -= 1`.
- ngược lại → **dùng quyền xóa duy nhất**: trả `is_pal_range(s, i+1, j) or is_pal_range(s, i, j-1)`.

**Đoạn 3:** thoát vòng lặp mà không lệch → đã là palindrome → `True`.

Chú ý: sau khi xóa, phần còn lại phải là palindrome **hoàn toàn** — không được xóa thêm, nên dùng `is_pal_range` (không có quyền xóa) chứ không gọi đệ quy lại hàm chính.

In [7]:
def valid_palindrome(s: str) -> bool:
    i, j = 0, len(s) - 1                         # đoạn 1
    while i < j:                                 # đoạn 2
        if s[i] == s[j]:
            i += 1
            j -= 1
        else:
            return is_pal_range(s, i + 1, j) or is_pal_range(s, i, j - 1)
    return True                                  # đoạn 3

for s in ["aba", "abca", "abc", "deeee", "eeccccbebaeeabebccceea", "a"]:
    print(f"{s!r:<26} -> {valid_palindrome(s)}")

'aba'                      -> True
'abca'                     -> True
'abc'                      -> False
'deeee'                    -> True
'eeccccbebaeeabebccceea'   -> False
'a'                        -> True


## 6. TRACE từng bước

Ví dụ khó: `s = "ebcbbececabbacecbbcbe"` — cặp lệch xuất hiện ở giữa và **chỉ một trong hai nhánh** thành công.

In [8]:
def trace_valid_palindrome(s: str) -> bool:
    i, j = 0, len(s) - 1
    print(f"s = {s!r} (n={len(s)})")
    while i < j:
        if s[i] == s[j]:
            print(f"  i={i:>2} {s[i]!r} == j={j:>2} {s[j]!r} -> thu hẹp")
            i += 1; j -= 1
        else:
            print(f"  i={i:>2} {s[i]!r} != j={j:>2} {s[j]!r} -> LỆCH, dùng quyền xóa")
            left = is_pal_range(s, i + 1, j)
            right = is_pal_range(s, i, j - 1)
            print(f"    bỏ s[{i}]: đoạn {s[i+1:j+1]!r} palindrome? {left}")
            print(f"    bỏ s[{j}]: đoạn {s[i:j]!r} palindrome? {right}")
            return left or right
    print("  không có cặp lệch -> đã là palindrome")
    return True

print("Kết quả:", trace_valid_palindrome("ebcbbececabbacecbbcbe"))

s = 'ebcbbececabbacecbbcbe' (n=21)
  i= 0 'e' == j=20 'e' -> thu hẹp
  i= 1 'b' == j=19 'b' -> thu hẹp
  i= 2 'c' == j=18 'c' -> thu hẹp
  i= 3 'b' == j=17 'b' -> thu hẹp
  i= 4 'b' == j=16 'b' -> thu hẹp
  i= 5 'e' != j=15 'c' -> LỆCH, dùng quyền xóa
    bỏ s[5]: đoạn 'cecabbacec' palindrome? True
    bỏ s[15]: đoạn 'ececabbace' palindrome? False
Kết quả: True


In [9]:
print("Kết quả:", trace_valid_palindrome("abca"))
print()
print("Kết quả:", trace_valid_palindrome("abc"))
print()
print("Kết quả:", trace_valid_palindrome("racecar"))

s = 'abca' (n=4)
  i= 0 'a' == j= 3 'a' -> thu hẹp
  i= 1 'b' != j= 2 'c' -> LỆCH, dùng quyền xóa
    bỏ s[1]: đoạn 'c' palindrome? True
    bỏ s[2]: đoạn 'b' palindrome? True
Kết quả: True

s = 'abc' (n=3)
  i= 0 'a' != j= 2 'c' -> LỆCH, dùng quyền xóa
    bỏ s[0]: đoạn 'bc' palindrome? False
    bỏ s[2]: đoạn 'ab' palindrome? False
Kết quả: False

s = 'racecar' (n=7)
  i= 0 'r' == j= 6 'r' -> thu hẹp
  i= 1 'a' == j= 5 'a' -> thu hẹp
  i= 2 'c' == j= 4 'c' -> thu hẹp
  không có cặp lệch -> đã là palindrome
Kết quả: True


Bảng trace cho `"abca"`:

| `i` | `j` | `s[i]` | `s[j]` | Quyết định |
|---:|---:|---|---|---|
| 0 | 3 | a | a | khớp → `i=1, j=2` |
| 1 | 2 | b | c | lệch → thử `s[2..2]="c"` ✔ hoặc `s[1..1]="b"` ✔ → `True` |

## 7. Code nộp LeetCode

### 7.1. Cách 1 — Hai con trỏ + hàm phụ (khuyên dùng)

In [10]:
class Solution:
    def validPalindrome(self, s: str) -> bool:
        def is_pal(i: int, j: int) -> bool:
            while i < j:
                if s[i] != s[j]:
                    return False
                i += 1
                j -= 1
            return True

        i, j = 0, len(s) - 1
        while i < j:
            if s[i] != s[j]:
                return is_pal(i + 1, j) or is_pal(i, j - 1)
            i += 1
            j -= 1
        return True

sol = Solution()
print(sol.validPalindrome("aba"), sol.validPalindrome("abca"), sol.validPalindrome("abc"))

True True False


### 7.2. Cách 2 — Hai con trỏ + slicing (ngắn, vẫn `O(n)`)
Khi lệch, cắt hai chuỗi ứng viên và so với bản đảo. Tốn thêm `O(n)` bộ nhớ nhưng code rất gọn.

In [11]:
class SolutionSlice:
    def validPalindrome(self, s: str) -> bool:
        i, j = 0, len(s) - 1
        while i < j:
            if s[i] != s[j]:
                a = s[i + 1:j + 1]       # bỏ s[i]
                b_ = s[i:j]              # bỏ s[j]
                return a == a[::-1] or b_ == b_[::-1]
            i += 1
            j -= 1
        return True

print(SolutionSlice().validPalindrome("abca"), SolutionSlice().validPalindrome("abc"))

True False


### 7.3. Cách 3 — Đệ quy với "ngân sách xóa" `budget`
`check(i, j, budget)`:
- `i >= j` → `True`.
- `s[i] == s[j]` → `check(i+1, j-1, budget)`.
- lệch: nếu `budget == 0` → `False`; ngược lại `check(i+1, j, budget-1) or check(i, j-1, budget-1)`.

Với `budget = 1` đây chính là bài 680. Phần khớp viết bằng vòng lặp để tránh đệ quy sâu `10^5` tầng.

In [12]:
class SolutionBudget:
    def validPalindrome(self, s: str) -> bool:
        def check(i: int, j: int, budget: int) -> bool:
            while i < j and s[i] == s[j]:      # đi qua phần khớp bằng vòng lặp
                i += 1
                j -= 1
            if i >= j:
                return True
            if budget == 0:
                return False
            return check(i + 1, j, budget - 1) or check(i, j - 1, budget - 1)
        return check(0, len(s) - 1, 1)

print(SolutionBudget().validPalindrome("abca"), SolutionBudget().validPalindrome("abc"))

True False


## 8. Vì sao đúng? (lập luận tham lam)

**Bổ đề 1:** nếu `s[i] == s[j]` (hai đầu khớp), thì `s` thành palindrome sau ≤ 1 lần xóa ⇔ `s[i+1..j-1]` thành palindrome sau ≤ 1 lần xóa.
- (⇐) xóa trong phần giữa xong, thêm lại hai đầu giống nhau vẫn là palindrome.
- (⇒) gọi `u = s[i+1..j-1]` (phần giữa), `c = s[i] = s[j]`. Nếu cách xóa không đụng hai đầu: chuỗi sau xóa có dạng `c + u' + c` là palindrome ⇒ `u'` (là `u` sau ≤ 1 lần xóa) là palindrome. Nếu cách xóa là xóa đầu `s[i]`: chuỗi còn lại `t = u + c` là palindrome ⇒ ký tự đầu `u[0]` bằng ký tự cuối `c`, và phần giữa `u[1:]` là palindrome ⇒ xóa `u[0]` khỏi `u` cũng cho palindrome. Trường hợp xóa `s[j]` đối xứng.
- Trực giác: hai đầu khớp thì "để nguyên" không bao giờ thiệt.

**Bổ đề 2:** tại cặp lệch đầu tiên `(i, j)`, mọi cách xóa thành công phải xóa `s[i]` hoặc `s[j]` (đã giải thích ở mục 4). Sau khi xóa, không còn quyền xóa → phần còn lại phải là palindrome nguyên vẹn.

Kết hợp hai bổ đề → thuật toán xét đúng và đủ mọi khả năng.

## 9. Độ phức tạp và so sánh

Gọi `n = len(s)`.

| Cách | Time | Space thêm | Ghi chú |
|---|---:|---:|---|
| Brute force xóa từng ký tự | `O(n²)` | `O(n)` | TLE với `n = 10^5` |
| Hai con trỏ + `is_pal(i, j)` | `O(n)` | `O(1)` | **Khuyên dùng** |
| Hai con trỏ + slicing | `O(n)` | `O(n)` | Code gọn |
| Đệ quy ngân sách `k` | `O(n·2^k)` | `O(k)` stack | `k = 1` → `O(n)`; tổng quát cho `k` nhỏ |

Với cách chính: vòng ngoài + tối đa 2 lần `is_pal` trên đoạn còn lại → tổng ≤ `3n/2` phép so sánh.

In [13]:
import time, random
rng = random.Random(680)
half = "".join(rng.choice("ab") for _ in range(50_000))
big = half + "z" + half[::-1]          # palindrome dài 100001
big_bad = big[:1000] + "q" + big[1000:]  # chèn 1 ký tự -> vẫn True
for name, f in [("two pointers", Solution().validPalindrome), ("slice", SolutionSlice().validPalindrome),
                ("budget", SolutionBudget().validPalindrome)]:
    t = time.perf_counter()
    r1, r2 = f(big), f(big_bad)
    print(f"{name:>13}: {r1}, {r2}  ({(time.perf_counter() - t) * 1000:.1f} ms)")

 two pointers: True, True  (5.8 ms)
        slice: True, True  (3.0 ms)
       budget: True, True  (5.7 ms)


## 10. Bộ test

1. Ba ví dụ đề.
2. Một ký tự, hai ký tự khác nhau (`"ab"` → `True`, xóa một cái).
3. Chỉ nhánh **bỏ trái** đúng / chỉ nhánh **bỏ phải** đúng.
4. Cần xóa **2** ký tự → `False`.
5. Ngẫu nhiên: so với brute force.

In [14]:
cases = [
    ("aba", True), ("abca", True), ("abc", False),
    ("a", True), ("ab", True), ("aa", True),
    ("deeee", True),                    # xóa 'd' (nhánh bỏ trái)
    ("eeeed", True),                    # xóa 'd' (nhánh bỏ phải)
    ("cbbcc", True),                    # lệch b/c ở giữa, chỉ nhánh bỏ phải đúng
    ("abccab", False),                  # cần xóa 2
    ("eeccccbebaeeabebccceea", False),
    ("aguokepatgbnvfqmgmlcupuufxoohdfpgjdmysgvhmvffcnqxjjxqncffvmhvgsymdjgpfdhooxfuupuculmgmqfvnbgtapekouga", True),
]
solvers = [Solution().validPalindrome, SolutionSlice().validPalindrome, SolutionBudget().validPalindrome,
           valid_palindrome, valid_palindrome_bruteforce]
for s, expected in cases:
    for f in solvers:
        assert f(s) == expected, (f, s)
    print(f"{s[:30]!r:<34} -> {expected} PASS")

'aba'                              -> True PASS
'abca'                             -> True PASS
'abc'                              -> False PASS
'a'                                -> True PASS
'ab'                               -> True PASS
'aa'                               -> True PASS
'deeee'                            -> True PASS
'eeeed'                            -> True PASS
'cbbcc'                            -> True PASS
'abccab'                           -> False PASS
'eeccccbebaeeabebccceea'           -> False PASS
'aguokepatgbnvfqmgmlcupuufxoohd'   -> True PASS


In [15]:
for _ in range(1000):
    n = rng.randint(1, 10)
    s = "".join(rng.choice("abc") for _ in range(n))
    exp = valid_palindrome_bruteforce(s)
    assert all(f(s) == exp for f in solvers), s
print("1000 chuỗi ngẫu nhiên (bảng chữ abc): mọi cách khớp brute force PASS")

1000 chuỗi ngẫu nhiên (bảng chữ abc): mọi cách khớp brute force PASS


## 11. Lỗi người mới hay gặp

### Lỗi 1: Chỉ thử **một** nhánh (luôn bỏ ký tự trái)
Sai với chuỗi mà chỉ nhánh **bỏ phải** đúng, ví dụ `"cuppucu"` (bỏ `'u'` cuối → `"cuppuc"`).

In [16]:
def wrong_only_left(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return is_pal_range(s, i + 1, j)        # SAI: quên nhánh bỏ phải
        i += 1; j -= 1
    return True

for s in ["cuppucu", "abca"]:
    print(f"{s!r}: sai = {wrong_only_left(s)}, đúng = {Solution().validPalindrome(s)}")

'cuppucu': sai = False, đúng = True
'abca': sai = True, đúng = True


### Lỗi 2: "Bỏ qua" cặp lệch rồi tiếp tục với **một biến đếm**, nhưng chọn bừa phía
Nhiều bạn viết `deleted += 1; i += 1` khi lệch rồi đi tiếp — tương đương lỗi 1, lại dễ đếm sai.

### Lỗi 3: Đệ quy gọi lại **hàm chính** sau khi xóa
`return validPalindrome(s[i+1:j+1]) or ...` cho phép xóa **thêm** lần nữa → chấp nhận cả chuỗi cần xóa 2 ký tự.

In [17]:
def wrong_recursive_main(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return wrong_recursive_main(s[i + 1:j + 1]) or wrong_recursive_main(s[i:j])   # SAI: xóa không giới hạn
        i += 1; j -= 1
    return True

print("sai:", wrong_recursive_main("abccab"), "| đúng:", Solution().validPalindrome("abccab"))
print("sai:", wrong_recursive_main("abc"), "| đúng:", Solution().validPalindrome("abc"))

sai: True | đúng: False
sai: True | đúng: False


### Lỗi 4: Lệch chỉ số khi cắt chuỗi
Đoạn `s[i..j]` (bao gồm `j`) trong slicing là `s[i:j+1]`. Viết `s[i+1:j]` là bỏ **cả hai** đầu.

### Lỗi 5: Dùng brute force cho `n = 10^5`
Chạy đúng trên ví dụ nhỏ nhưng TLE khi nộp.

In [18]:
s = "abca"; i, j = 1, 2
print("đúng (bỏ trái):", repr(s[i + 1:j + 1]), "| sai:", repr(s[i + 1:j]))

đúng (bỏ trái): 'c' | sai: ''


## 12. Mẹo nhận dạng pattern

- **Palindrome + so hai đầu** → hai con trỏ `i, j` tiến vào giữa.
- **"Được phép sai/xóa/sửa tối đa k lần"** → đi bình thường đến chỗ lỗi đầu tiên, rồi **rẽ nhánh** các lựa chọn với ngân sách giảm 1. Khi `k = 1` chỉ có 2 nhánh → `O(n)`.
- Muốn kiểm tra đoạn con nhiều lần → viết hàm `is_pal(i, j)` theo chỉ số thay vì cắt chuỗi.
- Bài cùng họ: 125 Valid Palindrome, 1216 Valid Palindrome III (xóa k ký tự — DP), 1328 Break a Palindrome.

## 13. Biến thể

### 13.1. Trả về **vị trí** cần xóa (hoặc `-1` nếu không cần, `None` nếu không thể)

In [19]:
def index_to_delete(s: str):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            if is_pal_range(s, i + 1, j):
                return i
            if is_pal_range(s, i, j - 1):
                return j
            return None
        i += 1; j -= 1
    return -1

for s in ["aba", "abca", "abc", "deeee"]:
    print(f"{s!r}: {index_to_delete(s)}")

'aba': -1
'abca': 1
'abc': None
'deeee': 0


### 13.2. Xóa tối đa `k` ký tự (Valid Palindrome III, k nhỏ)
Dùng lại `SolutionBudget` với `k` tùy ý. Với `k` lớn cần DP "dãy con palindrome dài nhất" (`O(n²)`).

In [20]:
def valid_palindrome_k(s: str, k: int) -> bool:
    def check(i, j, budget):
        while i < j and s[i] == s[j]:
            i += 1; j -= 1
        if i >= j:
            return True
        if budget == 0:
            return False
        return check(i + 1, j, budget - 1) or check(i, j - 1, budget - 1)
    return check(0, len(s) - 1, k)

print(valid_palindrome_k("abcdeca", 2), valid_palindrome_k("abbababa", 1), valid_palindrome_k("abccab", 2))

True True True


## 14. Bài tập (tự làm trước, đáp án ở cuối)

**Bài 1 — Trace tay:** với `s = "abecbea"`, ghi các cặp `(i, j)` được so, cặp lệch đầu tiên, và kết quả hai nhánh.

**Bài 2 — Đoán output:** `[valid_palindrome(x) for x in ["ba", "bab", "abbca", "tebbem"]]`.

**Bài 3 — TODO:** viết lại `is_pal_range` bằng `all()` và `range` (không dùng `while`).

In [21]:
def exercise_is_pal_range(s: str, i: int, j: int) -> bool:
    # TODO: so s[i + t] với s[j - t] cho t = 0, 1, ... (j - i) // 2
    pass

# assert exercise_is_pal_range("xabay", 1, 3) is True
# assert exercise_is_pal_range("abca", 0, 3) is False

**Bài 4 — TODO:** viết `exercise_valid_palindrome` (bài 680) không nhìn lại lời giải.

In [22]:
def exercise_valid_palindrome(s: str) -> bool:
    # TODO: hai con trỏ; khi lệch thử 2 nhánh
    pass

# assert exercise_valid_palindrome("abca") is True
# assert exercise_valid_palindrome("abc") is False

**Bài 5 — TODO (LeetCode 125):** kiểm tra palindrome sau khi **chỉ giữ chữ và số**, không phân biệt hoa thường (`str.isalnum()`, `str.lower()`), dùng hai con trỏ bỏ qua ký tự không hợp lệ.

In [23]:
def exercise_is_palindrome_alnum(s: str) -> bool:
    # TODO: while i < j: bỏ qua ký tự không alnum ở hai đầu; so sánh lower()
    pass

# assert exercise_is_palindrome_alnum("A man, a plan, a canal: Panama") is True
# assert exercise_is_palindrome_alnum("race a car") is False

**Bài 6 — Mini challenge:** đếm **số vị trí** `i` mà xóa `s[i]` cho ra palindrome (đề bài 680 chỉ hỏi có/không). Yêu cầu `O(n)` hoặc gần như vậy cho chuỗi không quá dài; có thể so với brute force để kiểm tra.

In [24]:
def exercise_count_deletions(s: str) -> int:
    # TODO (gợi ý dễ): brute force O(n^2) chấp nhận được cho n nhỏ; thử tối ưu nếu muốn
    pass

# assert exercise_count_deletions("abca") == 2      # xóa b hoặc c
# assert exercise_count_deletions("aaa") == 3
# assert exercise_count_deletions("abc") == 0

## 15. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | So `a`/`a`, `b`/`e`... | Lệch ở `i=1, j=5` | Thử `s[2..5]="ecbe"` và `s[1..4]="becb"` |
| 2 | `"ba"` xóa 1 ký tự | `"abbca"` lệch ở b/c | `"tebbem"`: lệch t/m, hai nhánh đều sai |
| 3 | Số cặp cần so là `(j - i + 1) // 2` | `all(... for t in range(...))` | `s[i + t] == s[j - t]` |
| 4 | Mục 5 | Hàm phụ `is_pal(i, j)` | `return is_pal(i+1, j) or is_pal(i, j-1)` |
| 5 | Hai vòng `while` nhỏ để nhảy qua ký tự rác | `s[i].isalnum()` | So `s[i].lower() != s[j].lower()` |
| 6 | Bắt đầu bằng brute force | `s[:i] + s[i+1:]` | Đếm `sum(...)` |

## 16. Đáp án tham khảo

**Bài 1:** `"abecbea"`: `(0,6)` a=a khớp; `(1,5)` b≠e lệch. Bỏ trái → `s[2..5] = "ecbe"` không palindrome. Bỏ phải → `s[1..4] = "becb"` không palindrome → `False`.

**Bài 2:** `[True, True, True, False]`.

In [25]:
print("Bài 1:", trace_valid_palindrome("abecbea"))
print("Bài 2:", [valid_palindrome(x) for x in ["ba", "bab", "abbca", "tebbem"]])

s = 'abecbea' (n=7)
  i= 0 'a' == j= 6 'a' -> thu hẹp
  i= 1 'b' != j= 5 'e' -> LỆCH, dùng quyền xóa
    bỏ s[1]: đoạn 'ecbe' palindrome? False
    bỏ s[5]: đoạn 'becb' palindrome? False
Bài 1: False
Bài 2: [True, True, True, False]


In [26]:
def answer_is_pal_range(s, i, j):
    return all(s[i + t] == s[j - t] for t in range((j - i + 1) // 2))


def answer_valid_palindrome(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return answer_is_pal_range(s, i + 1, j) or answer_is_pal_range(s, i, j - 1)
        i += 1; j -= 1
    return True


def answer_is_palindrome_alnum(s):
    i, j = 0, len(s) - 1
    while i < j:
        while i < j and not s[i].isalnum():
            i += 1
        while i < j and not s[j].isalnum():
            j -= 1
        if s[i].lower() != s[j].lower():
            return False
        i += 1; j -= 1
    return True


def answer_count_deletions(s):
    count = 0
    for i in range(len(s)):
        t = s[:i] + s[i + 1:]
        if t == t[::-1]:
            count += 1
    return count


assert answer_is_pal_range("xabay", 1, 3) is True
assert answer_is_pal_range("abca", 0, 3) is False
assert answer_valid_palindrome("abca") is True
assert answer_valid_palindrome("abc") is False
assert answer_is_palindrome_alnum("A man, a plan, a canal: Panama") is True
assert answer_is_palindrome_alnum("race a car") is False
assert answer_is_palindrome_alnum(" ") is True
assert answer_count_deletions("abca") == 2
assert answer_count_deletions("aaa") == 3
assert answer_count_deletions("abc") == 0
for _ in range(500):
    s = "".join(rng.choice("ab") for _ in range(rng.randint(1, 9)))
    assert answer_valid_palindrome(s) == valid_palindrome_bruteforce(s)
    assert (answer_count_deletions(s) > 0 or s == s[::-1]) == valid_palindrome_bruteforce(s)
print("Đáp án tham khảo: tất cả assert PASS")

Đáp án tham khảo: tất cả assert PASS


## 17. Template nhớ nhanh + checklist submit

```python
class Solution:
    def validPalindrome(self, s: str) -> bool:
        def is_pal(i, j):
            while i < j:
                if s[i] != s[j]:
                    return False
                i += 1; j -= 1
            return True

        i, j = 0, len(s) - 1
        while i < j:
            if s[i] != s[j]:
                return is_pal(i + 1, j) or is_pal(i, j - 1)   # thử bỏ trái HOẶC bỏ phải
            i += 1; j -= 1
        return True
```

**Câu thần chú:** *"Khớp thì đi tiếp; lệch lần đầu thì thử hai đường, sau đó không được xóa nữa."*

**Checklist:**
- [ ] Thử **cả hai** nhánh (bỏ `s[i]` và bỏ `s[j]`).
- [ ] Sau khi xóa, kiểm tra palindrome **nghiêm ngặt** (không xóa thêm).
- [ ] Chỉ số đoạn: bỏ trái là `[i+1, j]`, bỏ phải là `[i, j-1]`.
- [ ] Không dùng brute force `O(n²)` khi `n = 10^5`.
- [ ] Time `O(n)`, Space `O(1)`.